In [1]:
import os
import cv2
import  numpy as np
import random
import glob as glob
import torch
import torch.nn as nn
%pip install pexels_api_py
%pip install ultralytics
from pexelsapi.pexels import Pexels
import matplotlib.pyplot as plt

np.random.seed(42)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 98.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deutschland 0.4.2 requires numpy<2.0.0,>=1.26.2; python_version >= "3.12", but you have numpy 2.2.6 which is incompatible.
autodistill-yolov8 0.1.4 requires ultralytics==8.0.81, but you have ultralytics 8.3.241 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0d

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
^C


In [3]:

koala_video_path = "videos/koala_video1.mp4"
colibri_video_path = "videos/colibri_video1.mp4"
PEXELS_KEY = "fdmnaf3q0usOwaMzOIU0FWjE6CEgMvRYSfEEjywCn32H0c4cZSLN2PrI"
api = Pexels(PEXELS_KEY)

#search_results = api.search_photos(query="lion", orientation="landscape", size="large")
search_results

NameError: name 'search_results' is not defined

In [ ]:
import requests

def search_low_quality(search_results):
    low_res_video = None
    minWidth = 4000
    for video_file in search_results["videos"][0]["video_files"]:
        if video_file["width"] < minWidth:
            minWidth = video_file["width"]
            low_res_video = video_file["link"]

    return low_res_video


video_link = search_low_quality(search_results)
print(video_link)

In [ ]:
import requests

def download_video(url, save_path):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print(f"Video downloaded successfully to {save_path}")

    except Exception as e:
        print(f"Video could not be loaded: {e}")

# Use it like this:
download_video(video_link, "./content/videos/lion_video1.mp4")

In [ ]:
train = True
epochs = 25

## Dataset creation

In [ ]:
%pip install autodistill autodistill-grounding-dino autodistill-yolov8 supervision

In [ ]:
img_tags = ["wildlife", "nature", "forest animals", "safari", "wilderness", "national park"]
species_tags = ["pigeon", "fox", "wolf", "bear", "frog" "roe deer", "wild boar", "red squirrel", "common buzzard"]

all_tags = img_tags + species_tags
all_tags

In [ ]:
all_tags2 = ["elephant", "polar bear", "lion", "gorilla", "waterfowl",
             "butterfly", "zebra", "turtle", "bald eagle"]

In [ ]:
import requests
import os

def download_photo(url, idx):
    folder = "content/images"
    os.makedirs(folder, exist_ok=True)
    file_path = os.path.join(folder, f"image_{idx}.jpg")

    try:
        response = requests.get(url, timeout=10)
        # Only save if the request was successful (Status 200)
        if response.status_code == 200:
            with open(file_path, "wb") as f:
                f.write(response.content)
        else:
            print(f"Failed to download image {idx}. Status: {response.status_code}")
    except Exception as e:
        print(f"Error downloading image {idx}: {e}")

# --- Main Loop ---
"""
photos_count = 0
for tag in all_tags2:
    # Use the pexels_api library search
    results = api.search_photos(query=tag, orientation="landscape", per_page=5)

    # The 'results' object is a dictionary, not a direct iterable of photos.
    # We need to access the 'photos' key, which contains a list of photo dictionaries.
    for photo_dict in results["photos"]:
        # Each 'photo_dict' is a dictionary, so access its keys
        url = photo_dict["src"]["original"]
        download_photo(url, photos_count)
        photos_count += 1 """

#print(f"✅ {photos_count} photos saved successfully in 'content/images/'")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
all_tags = all_tags + all_tags2
all_tags

## Label Images using Autodistill autolabeling


In [ ]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

ontology_dict = {
    f"Phot of a {tag}": tag for tag in all_tags
}
ontology = CaptionOntology(ontology_dict)

base_model = GroundingDINO(ontology=ontology)
dataset = base_model.label(
    input_folder="content/images",
    extension=".jpg",
    output_folder="content/dataset"
)

In [ ]:
%pip install roboflow

In [ ]:
# Delete the old file if it exists
if os.path.exists("yolov8n.pt"):
    os.remove("yolov8n.pt")

# Redownload fresh
yolo = YOLO("yolov8n.pt")

## YOLO Model training

In [ ]:
import os
# This MUST be set before importing YOLO
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

# 1. Update Ultralytics to the latest version
!pip install -U ultralytics --quiet

from ultralytics import YOLO

# 2. Re-download a clean model file to fix potential corruption
if os.path.exists("yolov8n.pt"):
    !rm yolov8n.pt

# 3. Load and train
model = YOLO("yolov8n.pt")
model.train(data="/content/dataset/data.yaml", epochs=50, imgsz=640)